# A Hierarchical Mixture Model of Bimodal Honeybee Waggle-Dances at Low Solar Elevations
---

## Details

&ensp;AUTHOR:&emsp;   James Foster &emsp; 2026 10 20

MODIFIED:&emsp; James Foster &emsp; 2026 10 20

DESCRIPTION:&emsp; Fit hierarchical von Mises models to honeybee waggle-run angles relative to the feeder at low solar elevations (near sunset), when dances on a horizontal comb are often bimodal (tending towards axial symmetry).
A unimodal von Mises model is compared with mixtures of two von Mises distributions with opposite means ($\mu$ and $\mu + \pi$) for vertical and horizontal dances.
The proportion of waggle runs in the primary mode ($w$) is modelled with its own formula, so that the hypothesis that horizontal dances are more often bimodal than vertical dances can be tested directly.
Modified from [HS_circGLMM_fit_4cond.ipynb](HS_circGLMM_fit_4cond.ipynb) and [locust_direction_mixture_model.ipynb](https://github.com/jgraving/unwrap/blob/main/notebooks/locust_direction_mixture_model.ipynb) from Jake Graving's `unwrap` package, using the `BimodalVonMises`/`VonMisesMixture` distributions from `unwrap.distributions` as a custom `Bambi` likelihood.

INPUTS:&emsp;    `0Data/hs_dance_reorg.csv`

OUTPUTS:&emsp;  `0Data/hs_dance_lowel_VH.feather`, `2Results/unwrap_models/HS_lowel_*.nc`, `Plots/HS_lowel_VH_model_comparison.pdf`, `Plots/HS_lowel_VH_bimodality.pdf`

REFERENCES:
- Sayin S, ... Graving JM, $\textit{et~al}$. (2025)
The behavioral mechanisms governing collective motion in swarming locusts.
$\textit{Science}$ 387,995-1000
DOI:[10.1126/science.adq7832](https://doi.org/10.1126/science.adq7832)

- Graving JM, Sayin S, Costelloe BR, Goldshtein A & Foster JJ.
Unwrapping circular statistics with Bayesian multilevel models.

- Capretto, T., Piho, C., Kumar, R., Westfall, J., Yarkoni, T., & Martin, O. (2022)
`Bambi`: A simple interface for fitting Bayesian linear models in Python.
*Journal of Statistical Software* 103(15), 1-29. doi:10.18637/jss.v103.i15

- Seyboldt, A., Carlson, E. L., & Carpenter, B. (2026).
Preconditioning Hamiltonian Monte Carlo by minimizing Fisher Divergence.
*arXiv* preprint arXiv:2603.18845.

- Stephens, M. (2000).
Dealing with label switching in mixture models.
*Journal of the Royal Statistical Society B* 62(4), 795-809.

---

# First set up the required packages
Same environment as [HS_circGLMM_fit_4cond.ipynb](HS_circGLMM_fit_4cond.ipynb) (`HS-env`); skip this cell if it is already installed.

In [ ]:
import sys

# 20261014 - Pinned unwrap (later commits require bambi>=0.15), nutpie and pyarrow (>=25 requires NumPy 2),
#            ipython & ipykernel (later versions show matplotlib figures as text only)
requirements = """
numpy>=1.26.4,<2.0.0
pandas==2.2.2
scipy==1.13.1
arviz==0.19.0
bambi==0.14.0
pymc==5.16.2
matplotlib==3.8.4
formulae==0.5.4
numba>=0.59.1,<=0.60.0
nutpie==0.13.2
netcdf4>=1.6.5,<1.7.0
h5py>=3.10.0,<4.0.0
git+https://github.com/jgraving/unwrap.git@3e701175b665a12f4a0dcda6643aef9ea4b7aecf
pyarrow==24.0.0
ipython==9.13.0
ipykernel==6.29.5
"""

# Write it to a file
with open("nutpie_unwrap_requirements.txt", "w") as f:
    f.write(requirements.strip() + "\n")

# Install everything from the file (on Windows "{sys.executable}" needs to be in quotes)
!"{sys.executable}" -m pip install -r nutpie_unwrap_requirements.txt --no-cache

%set_env PYTENSOR_FLAGS=blas__ldflags='-llapack -lblas -lcblas'

## Sampler settings
Shared by all models.
`nutpie` handles these mixture models without technical problems (no divergences in test fits).
The known difficulty with mixtures is a multimodal posterior (label switching, see below), which affects every MCMC sampler.
If chains disagree, more chains make it easier to see how often each labelling is visited, and `"pymc"` can be used for comparison.
In test fits, each model took roughly 5 minutes per 600 iterations.

In [ ]:
sampler = "nutpie" # or "pymc"
n_draws = 1000
n_tune = 1000
n_chains = 4
target_accept = 0.95
random_seed = 20261020

# Set up custom functions
## An axial von Mises mixture as a `Bambi` family
Each waggle run is either in the primary mode, $\mu$, with probability $w$, or in the opposite direction, $\mu + \pi$, with probability $1-w$, and both modes share the same concentration, $\kappa$:

$$
f(\theta \mid \mu, \kappa, w) = w\,\mathcal{M}(\theta \mid \mu, \kappa) + (1-w)\,\mathcal{M}(\theta \mid \mu + \pi, \kappa)
$$

- $w = 1$: unimodal, all runs towards $\mu$
- $w = 0.5$: axially symmetric, runs equally often towards $\mu$ and $\mu + \pi$
- $w = 0$: unimodal but reversed, all runs towards $\mu + \pi$

$w$ has a logit link, so it can take its own formula with fixed and random effects, as $\mu$ and $\kappa$ do.
Fixing the second mode at $\mu + \pi$ follows the axial tendency seen in `HS_ExploratoryPlots.Rmd` and avoids a free mode separation, which would add another source of label switching.

*N.B.* The likelihood is unchanged by swapping the labels of the two modes ($\mu \to \mu + \pi$, $w \to 1 - w$), so $\mu$ and $w$ are not separately identified.
This is addressed in post-processing (`relabel()`), as for the locust example, and hypotheses are tested on quantities that are unaffected by labelling.

In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import pytensor.tensor as pt
import bambi as bmb
import arviz as az
import matplotlib.pyplot as plt

from scipy.special import expit
from scipy.stats import vonmises, circmean
from unwrap.link.bambi import LINKS
from unwrap.link.numpy import inverse_softplus, shifted_modulo, softplus
from unwrap.prior import CircularUniformPrior
from unwrap.distributions import VonMisesMixture


class AxialVonMises:
    """
    Mixture of two von Mises distributions with opposite means (mu and mu + pi) and a shared
    concentration (kappa), for use as a custom Bambi likelihood.

    w is the probability of the primary mode (mu): w = 1 is unimodal, w = 0.5 axially symmetric.
    Called with a name inside a model (response distribution) or via .dist() (log-likelihood,
    posterior predictive).
    """
    @staticmethod
    def _components(mu, kappa, w):
        mu = pt.as_tensor_variable(mu)
        mu_rev = pt.arctan2(-pt.sin(mu), -pt.cos(mu)) # mu + pi, wrapped to [-pi, pi]
        return (pt.stack([w, 1 - w], axis=-1),
                pt.stack([mu, mu_rev], axis=-1),
                pt.stack([kappa, kappa], axis=-1))

    def __new__(cls, name, mu, kappa, w, **kwargs):
        return VonMisesMixture(name, *cls._components(mu, kappa, w), **kwargs)

    @classmethod
    def dist(cls, mu, kappa, w, **kwargs):
        return VonMisesMixture.dist(*cls._components(mu, kappa, w), **kwargs)


axial_family = bmb.Family(
    "axial_vonmises",
    likelihood=bmb.Likelihood("AxialVonMises",
                              params=["mu", "kappa", "w"],
                              parent="mu",
                              dist=AxialVonMises),
    link={"mu": LINKS["shifted_modulo"], "kappa": LINKS["softplus"], "w": "logit"},
)
link_vonmises = {"mu": LINKS["shifted_modulo"], "kappa": LINKS["softplus"]}

## Post-processing and plotting functions
- `relabel()`: label the mode nearest the feeder (0°) as primary, swapping $\mu \to \mu + \pi$ and $w \to 1 - w$ where $\cos\mu < 0$. The mixture density is unchanged.
- `bimodality()`: $B = 1 - |2w - 1|$, which is 0 for a unimodal distribution (either direction) and 1 for an axially symmetric one. $B$ is unaffected by labelling.
- `population_params()`: population-level $\mu$, $\kappa$ and $w$ (random effects at zero) for a condition, given its fixed effects.
- `axial_pdf()`: density of the axial mixture.
- `fit_model()`: fit with the shared sampler settings, compute the log-likelihood and save.

In [ ]:
def relabel(mu, w):
    """Label the mode nearest the feeder (0°) as primary; the mixture density is unchanged."""
    flip = np.cos(mu) < 0
    return shifted_modulo(xr.where(flip, mu + np.pi, mu)), xr.where(flip, 1 - w, w)


def bimodality(w):
    """Bimodality index: 0 for unimodal (w = 0 or 1), 1 for axially symmetric (w = 0.5)."""
    return 1 - np.abs(2 * w - 1)


def population_params(post, effects=()):
    """
    Population-level mu, kappa and w (random effects at zero) for the condition defined by the
    fixed effects in `effects` (e.g. () for the reference, ('H',) for horizontal), relabelled so
    that mu is the mode nearest the feeder. For unimodal models w = 1.
    """
    mu = shifted_modulo(post['Intercept'] + sum(post[e] for e in effects))
    kappa = softplus(post['kappa_Intercept'] + sum(post['kappa_' + e] for e in effects))
    if 'w_Intercept' in post:
        w = expit(post['w_Intercept'] + sum(post['w_' + e] for e in effects if 'w_' + e in post))
    else:
        w = xr.ones_like(kappa)
    mu, w = relabel(mu, w)
    return mu, kappa, w


def axial_pdf(theta, mu, kappa, w):
    """Density of the axial von Mises mixture (w = 1 gives a von Mises)."""
    return (w * vonmises.pdf(theta, kappa=kappa, loc=mu) +
            (1 - w) * vonmises.pdf(theta, kappa=kappa, loc=mu + np.pi))


def fit_model(model, label):
    """Fit with the shared sampler settings, attach the log-likelihood and save."""
    idata = model.fit(
        draws=n_draws,
        tune=n_tune,
        chains=n_chains,
        nuts_sampler=sampler,
        target_accept=target_accept,
        random_seed=random_seed,
    )
    model.compute_log_likelihood(idata)
    os.makedirs("2Results/unwrap_models", exist_ok=True)
    az.to_netcdf(idata, f"2Results/unwrap_models/HS_lowel_{label}.nc")
    return idata

# Prepare the data
## Import and format the data
Load the preprocessed data from CSV and select:
- **low solar elevations**: `sun_el` $< 15°$ (as in `HS_ExploratoryPlots.Rmd`), i.e. dances recorded in the hour before sunset
- dances in the **vertical** or **horizontal** conditions

Waggle-run angles relative to the feeder bearing (`feeder_angle`) are already in radians.
The vertical condition is the reference and `H` is the effect of horizontal orientation.

In [ ]:
# Load the file
file_path = '0Data/hs_dance_reorg.csv'
df_all = pd.read_csv(file_path, sep=',')

# Conditions to include, reference (vertical) first
conditions_incl = ['vertical', 'horizontal']

# Select low solar elevations and the conditions of interest
lowel_max = 15 # degrees, as in HS_ExploratoryPlots.Rmd
df = df_all[
    (df_all['sun_el'] < lowel_max) &
    (df_all['condition'].isin(conditions_incl))
].copy()

# Effect of horizontal orientation (vertical is the reference)
df['H'] = (df['condition'] == 'horizontal').astype(int)
# Rename waggle run to use capitalised label
df['Run'] = df['waggle_run']

# Convert the feeder angle column to numeric (already radians)
df['angle'] = pd.to_numeric(df['feeder_angle'], errors='coerce')  # Convert non-numeric values (e.g., 'NA') to NaN
# Sun azimuth relative to the feeder, for plotting (radians)
df['SunAz'] = np.deg2rad(df['sun_az'] - df['feeder_az'])

# Normalise all angles to be between -pi and pi (should be already)
df['angle'] = df['angle'].apply(lambda x: shifted_modulo(x) if pd.notnull(x) else x)
df['SunAz'] = df['SunAz'].apply(lambda x: shifted_modulo(x) if pd.notnull(x) else x)

# Keep only the relevant columns, dropping missing angles so that rows match the model observations
df = df[['angle', 'ID', 'ID_d', 'condition', 'H', 'Run', 'SunAz', 'sun_el']].dropna(subset=['angle'])
df.reset_index(drop=True, inplace=True)

# Rename columns: change 'ID' to 'Individual' and 'ID_d' to 'Dance'
df = df.rename(columns={'ID': 'Individual', 'ID_d': 'Dance'})
df['Individual'] = df['Individual'].astype('category')
df['Dance'] = df['Dance'].astype('category')

# Keep the original labels for reference, then simplify to codes
labels = df[['Individual', 'Dance']].astype(str).rename(columns=lambda c: c + '_label')
df['Individual'] = df['Individual'].cat.codes + 1
df['Dance'] = df['Dance'].cat.codes + 1
df = pd.concat([df, labels], axis=1)

#save to align with raw data in R
df.to_feather('0Data/hs_dance_lowel_VH.feather')

# Report sample sizes
print(df.groupby('condition').agg(runs=('angle', 'size'),
                                  dances=('Dance', 'nunique'),
                                  individuals=('Individual', 'nunique')).loc[conditions_incl])
print('Individuals in both conditions:',
      len(set(df.loc[df['H'] == 0, 'Individual']) & set(df.loc[df['H'] == 1, 'Individual'])))
df.head()

## Inspect the data
Left: all waggle runs in each condition.
Right: for each dance, the proportion of waggle runs within 90° of the feeder direction.
Unimodal dances towards the feeder lie at 1, reversed dances at 0, and bimodal dances in between.

In [ ]:
# Colours for each condition, as in HS_ExploratoryPlots.Rmd
cond_cols = {
    'vertical': 'darkred',
    'horizontal': 'darkgreen',
}

fig = plt.figure(figsize=(14, 6))
bins = np.linspace(-np.pi, np.pi, 37) # 10° bins

for i, (cond, col) in enumerate(cond_cols.items()):
    data_c = df[df['condition'] == cond]
    # Rose plot of all waggle runs
    ax = fig.add_subplot(2, 2, 2 * i + 1, projection='polar')
    ax.set_theta_direction(-1) # reverse to clockwise
    ax.set_theta_zero_location('N') # place 0° (feeder) at the top
    ax.hist(data_c['angle'], bins=bins, color=col, alpha=0.7)
    ax.set_title(f"{cond.capitalize()} ({len(data_c)} runs, {data_c['Dance'].nunique()} dances)", pad=15)
    # Proportion of runs towards the feeder in each dance
    ax = fig.add_subplot(2, 2, 2 * i + 2)
    prop_feeder = data_c.groupby('Dance')['angle'].apply(lambda x: np.mean(np.cos(x) > 0))
    ax.hist(prop_feeder, bins=np.linspace(0, 1, 11), color=col, alpha=0.7)
    ax.set_xlabel('Proportion of waggle runs within 90° of feeder')
    ax.set_ylabel('Dances')

plt.tight_layout()
plt.show()

# Priors
Mostly following `HS_circGLMM_fit_4cond.ipynb`, with two changes to make the two modes identifiable.

**1. Individual and dance offsets in $\mu$ are Normal rather than von Mises.**
With von Mises random effects, a reversed dance can be represented either by a low $w$ for that dance or by shifting its $\mu$ by 180° and keeping $w$ high.
In a first fit, the dance-level concentration collapsed from its prior median of 30 to about 2, where a 180° offset is cheap, and chains disagreed on which representation to use.
Normal (unwrapped) offsets with a half-normal scale still allow substantial variation between individuals and dances, but put a 180° offset several standard deviations out, so reversals must be explained by $w$.

**2. The mixture weight for the reference (vertical) condition is constrained to favour the primary mode** ($\text{logit}(w) > 0$).
All 34 vertical dances at low elevation have most waggle runs within 90° of the feeder, so the primary mode is defined as the one that vertical dances favour.
This removes the global label switch ($\mu \to \mu + \pi$, $w \to 1 - w$), which was an exact symmetry with a circular uniform `Intercept` and a symmetric prior on `w_Intercept`.
`relabel()` is kept as a check: it should rarely change anything.

In [ ]:
# Individual-level and dance-level effects for μ: Normal (unwrapped) offsets, so that a 180° offset is
# implausible and reversed dances are explained by w instead.
# Scale of the half-normal prior on their SD (radians): 0.5 puts 180° at > 6 SD for a typical SD,
# 1.0 allows larger drifts in direction (90° ≈ 1.6 SD) but makes a 180° offset less implausible (≈ 3 SD)
mu_re_scale = 0.5
prior_mu_ind = bmb.Prior(
    "Normal", mu=0, sigma=bmb.Prior("HalfNormal", sigma=mu_re_scale)
)
prior_mu_dance = bmb.Prior(
    "Normal", mu=0, sigma=bmb.Prior("HalfNormal", sigma=mu_re_scale)
)
# Fixed effect of condition for μ: weak, centred on no change
prior_mu_effect = bmb.Prior("VonMises", mu=0, kappa=0.5)
# Fixed effects for κ
prior_kappa_intercept = bmb.Prior("Normal", mu=inverse_softplus(3), sigma=3.0)
prior_kappa_effect = bmb.Prior("Normal", mu=0, sigma=1.0)
# Fixed effects for w (logit scale): the reference (vertical) condition favours the primary mode, logit(w) > 0,
# median w ≈ 0.9; this fixes which mode is labelled primary
prior_w_intercept = bmb.Prior("TruncatedNormal", mu=2, sigma=1.5, lower=0)
prior_w_effect = bmb.Prior("Normal", mu=0, sigma=1.5)
# Random effects for κ and w
prior_re = bmb.Prior(
    "Normal", mu=0, sigma=bmb.Prior("HalfStudentT", nu=3, sigma=1.0)
)

# Shared random-effect priors
priors_mu_re = {"1|Individual": prior_mu_ind, "1|Dance": prior_mu_dance}
priors_re = {"1|Individual": prior_re, "1|Dance": prior_re}

# Alternative hypotheses: are horizontal dances more often bimodal?
Individuals and dances (nested within individuals) receive random intercepts for every parameter, so each dance can have its own direction, concentration and mixture weight.
Random slopes for `H` are omitted for this draft (only 12 individuals danced in both conditions).

0. **Unimodal**: condition affects mean direction and concentration only
    - $\mu \sim$ H + (1 |Individual) + (1 |Dance)
    - $\kappa \sim$ H + (1 |Individual) + (1 |Dance)
1. **Bimodal, same mixing**: dances may be bimodal, equally often in both conditions
    - as model 0, plus $w \sim$ 1 + (1 |Individual) + (1 |Dance)
2. **Bimodal, condition-dependent mixing**: the mixture weight differs between conditions
    - as model 0, plus $w \sim$ H + (1 |Individual) + (1 |Dance)

Comparing model 2 with model 1 tests whether the proportion of waggle runs in each mode differs between conditions.

In [ ]:
formula_mu = "angle ~ H + (1 |Individual) + (1 |Dance)"
formula_kappa = "kappa ~ H + (1 |Individual) + (1 |Dance)"

priors_vh_uni = {
    "Intercept": CircularUniformPrior(),
    "H": prior_mu_effect,
    **priors_mu_re,
    "kappa": {"Intercept": prior_kappa_intercept, "H": prior_kappa_effect, **priors_re},
}

#model 0
model_vh_uni = bmb.Model(
    bmb.Formula(formula_mu, formula_kappa),
    df,
    family="vonmises",
    priors=priors_vh_uni,
    link=link_vonmises,
    center_predictors=False,
)

#model 1
model_vh_axial_w1 = bmb.Model(
    bmb.Formula(formula_mu, formula_kappa, "w ~ 1 + (1 |Individual) + (1 |Dance)"),
    df,
    family=axial_family,
    priors={**priors_vh_uni,
            "w": {"Intercept": prior_w_intercept, **priors_re}},
    center_predictors=False,
)

#model 2
model_vh_axial_wH = bmb.Model(
    bmb.Formula(formula_mu, formula_kappa, "w ~ H + (1 |Individual) + (1 |Dance)"),
    df,
    family=axial_family,
    priors={**priors_vh_uni,
            "w": {"Intercept": prior_w_intercept, "H": prior_w_effect, **priors_re}},
    center_predictors=False,
)

print(model_vh_axial_wH)

## Fit the models

In [ ]:
idata_vh_uni = fit_model(model_vh_uni, "VH_0_unimodal")
idata_vh_axial_w1 = fit_model(model_vh_axial_w1, "VH_1_axial_w1")
idata_vh_axial_wH = fit_model(model_vh_axial_wH, "VH_2_axial_wH")

## Check convergence heuristics
If chains visit different labellings, the `Intercept`, `H` and `w_` parameters (and the scales of their random effects) will show poor convergence ($\hat{R} > 1.01$) and disagreement between chain means.
The priors above should prevent this for the reference condition, but a chain could still settle on the swapped labelling for the horizontal condition alone (horizontal $\mu$ near 180° with a strongly negative `w_H`).
The proportion of swapped draws per chain is printed below; it should be 0 for all chains.

In [ ]:
az.summary(
    idata_vh_axial_wH,
    var_names=["Intercept", "H", "kappa_Intercept", "kappa_H", "w_Intercept", "w_H",
               "1|Individual_sigma", "1|Dance_sigma",
               "kappa_1|Individual_sigma", "kappa_1|Dance_sigma",
               "w_1|Individual_sigma", "w_1|Dance_sigma"],
    circ_var_names=["Intercept", "H"],
)

In [ ]:
post_vh = idata_vh_axial_wH.posterior
mu_v, kappa_v, w_v = population_params(post_vh)
mu_hz, kappa_hz, w_hz = population_params(post_vh, effects=('H',))

print('Proportion of draws with swapped labels in each chain:')
print('  vertical:  ', (np.cos(shifted_modulo(post_vh['Intercept'])) < 0).mean('draw').values.round(2))
print('  horizontal:', (np.cos(shifted_modulo(post_vh['Intercept'] + post_vh['H'])) < 0).mean('draw').values.round(2))

# Relabelled population-level parameters (mean directions in radians, relative to the feeder)
derived_vh = xr.Dataset({
    'mu_vertical': mu_v,
    'mu_horizontal': mu_hz,
    'kappa_vertical': kappa_v,
    'kappa_horizontal': kappa_hz,
    'w_vertical': w_v,
    'w_horizontal': w_hz,
})
for cond, mu_c in {'vertical': mu_v, 'horizontal': mu_hz}.items():
    print(f"Mean direction, {cond}: {np.rad2deg(circmean(mu_c.values.ravel(), high=np.pi, low=-np.pi)):.1f}°")
az.summary(derived_vh, circ_var_names=['mu_vertical', 'mu_horizontal'])

In [ ]:
az.plot_trace(
    idata_vh_axial_wH,
    var_names=["Intercept", "H", "kappa_Intercept", "kappa_H", "w_Intercept", "w_H",
               "1|Individual_sigma", "1|Dance_sigma",
               "kappa_1|Individual_sigma", "kappa_1|Dance_sigma",
               "w_1|Individual_sigma", "w_1|Dance_sigma"],
    circ_var_names=["Intercept", "H"],
)
plt.tight_layout()
plt.show()

## Inspect dance-level mixture weights
Posterior mean $w$ for each dance against the observed proportion of its waggle runs within 90° of the (relabelled) population primary mode for its condition.
Points near the diagonal indicate that the dance-level random effects on $w$ capture each dance's balance between modes.

In [ ]:
# The model's w is constant within each dance, so take the first observation of each dance
first_obs = df.groupby('Dance').head(1).index.values
dance_cond = df.loc[first_obs, 'condition'].values
w_dance = post_vh['w'].isel(__obs__=first_obs)
# Relabel each draw consistently with the population-level primary mode of each dance's condition
mu_raw_cond = shifted_modulo(post_vh['Intercept'] + post_vh['H'] * xr.DataArray((dance_cond == 'horizontal').astype(int), dims='__obs__'))
w_dance = xr.where(np.cos(mu_raw_cond) < 0, 1 - w_dance, w_dance)

fig, ax = plt.subplots(figsize=(6, 5))
for cond, mu_c in {'vertical': mu_v, 'horizontal': mu_hz}.items():
    mu_primary = circmean(mu_c.values.ravel(), high=np.pi, low=-np.pi)
    idx = np.where(dance_cond == cond)[0]
    prop_primary = [np.mean(np.cos(df.loc[df['Dance'] == d, 'angle'] - mu_primary) > 0)
                    for d in df.loc[first_obs[idx], 'Dance']]
    ax.scatter(prop_primary, w_dance.isel(__obs__=idx).mean(('chain', 'draw')),
               color=cond_cols[cond], alpha=0.7,
               label=f'{cond.capitalize()} (primary mode {np.rad2deg(mu_primary):.0f}°)')
ax.plot([0, 1], [0, 1], color='gray', linestyle='--')
ax.set_xlabel('Observed proportion within 90° of primary mode')
ax.set_ylabel('Posterior mean $w$')
ax.legend()
plt.show()

## Model comparison
Leave-one-out cross-validation (LOO-CV) of each waggle run.
LOO is unaffected by label switching, since the likelihood is the same under either labelling.

*N.B.* With dance-level random effects, many waggle runs are influential (Pareto $\hat{k} > 0.7$ for ~5% of runs in test fits), so the PSIS-LOO estimates may be optimistic.
If this is the case, K-fold cross-validation leaving out whole dances would be more reliable (and closer to the question of predicting new dances).

In [ ]:
import matplotlib as mpl

# Ensure text is output as editable TrueType text elements in PDFs
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'Computer Modern Sans Serif']

loo_vh = {
    'Unimodal': az.loo(idata_vh_uni),
    'Bimodal, same mixing': az.loo(idata_vh_axial_w1),
    'Bimodal, condition-dependent mixing': az.loo(idata_vh_axial_wH),
}
comparison_vh = az.compare(loo_vh, ic="loo")
print('\n----- Model Comparison Ranking, vertical vs horizontal -----\n')
print(comparison_vh.round(2))

ax_compare = az.plot_compare(comparison_vh)
ax_compare.get_figure().savefig("Plots/HS_lowel_VH_model_comparison.pdf", format="pdf", bbox_inches="tight")

## Test the hypothesis
**Hypothesis**: horizontal dances are more likely to be bimodal than vertical dances.

Using model 2, bimodality is measured by $B = 1 - |2w - 1|$, which does not depend on which mode is labelled primary:
1. **Population level**: $B$ for a typical dance (random effects at zero) in each condition, $\Delta B = B_\text{horizontal} - B_\text{vertical}$.
2. **Dance level**: mean $B$ across the observed dances in each condition, which includes the dance and individual random effects.

The posterior probability $P(\Delta B > 0)$ is the support for the hypothesis.
After relabelling, $w_\text{horizontal} < w_\text{vertical}$ indicates more waggle runs in the reverse mode on the horizontal comb.

In [ ]:
# 1. Population level
B_vert = bimodality(w_v)
B_hor = bimodality(w_hz)
dB_pop = B_hor - B_vert

# 2. Dance level (first_obs and dance_cond from the dance-level inspection above)
B_dance =bimodality(post_vh['w'].isel(__obs__=first_obs))
B_dance_vert = B_dance.isel(__obs__=np.where(dance_cond == 'vertical')[0]).mean('__obs__')
B_dance_hor = B_dance.isel(__obs__=np.where(dance_cond == 'horizontal')[0]).mean('__obs__')
dB_dance = B_dance_hor - B_dance_vert

hypothesis = xr.Dataset({
    'B_vertical_pop': B_vert,
    'B_horizontal_pop': B_hor,
    'dB_pop': dB_pop,
    'B_vertical_dances': B_dance_vert,
    'B_horizontal_dances': B_dance_hor,
    'dB_dances': dB_dance,
    'w_horizontal_minus_vertical': w_hz - w_v,
})
display(az.summary(hypothesis, hdi_prob=0.95))

print(f"P(ΔB > 0), population level: {(dB_pop > 0).mean().item():.3f}")
print(f"P(ΔB > 0), observed dances:  {(dB_dance > 0).mean().item():.3f}")
print(f"P(w_horizontal < w_vertical): {(w_hz < w_v).mean().item():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, (B_v, B_h, dB, title) in zip(axes, [
        (B_vert, B_hor, dB_pop, 'Population level (typical dance)'),
        (B_dance_vert, B_dance_hor, dB_dance, 'Mean across observed dances')]):
    bins_B = np.linspace(0, 1, 41)
    ax.hist(B_v.values.ravel(), bins=bins_B, color=cond_cols['vertical'], alpha=0.6, density=True, label='Vertical')
    ax.hist(B_h.values.ravel(), bins=bins_B, color=cond_cols['horizontal'], alpha=0.6, density=True, label='Horizontal')
    ax.set_xlabel('Bimodality, $B = 1 - |2w - 1|$')
    ax.set_ylabel('Posterior density')
    ax.set_title(f"{title}\nP(ΔB > 0) = {(dB > 0).mean().item():.3f}")
    ax.legend()
plt.tight_layout()
fig.savefig("Plots/HS_lowel_VH_bimodality.pdf", format="pdf", bbox_inches="tight")
plt.show()

# Visualise population-level densities
Population-level densities (random effects at zero) from the unimodal model (0) and the condition-dependent mixture (2), against the observed waggle-run angles.
The mixture density does not depend on labelling.

In [ ]:
theta = np.linspace(-np.pi, np.pi, 361)
theta_deg = np.rad2deg(theta)
post_uni = idata_vh_uni.posterior

def density_band(mu, kappa, w):
    """Posterior median and 95% interval of the population-level density."""
    pdf = axial_pdf(theta[None, :],
                    mu.values.ravel()[:, None],
                    kappa.values.ravel()[:, None],
                    w.values.ravel()[:, None])
    return np.median(pdf, axis=0), np.percentile(pdf, [2.5, 97.5], axis=0)

fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for ax, (cond, effs) in zip(axes, {'vertical': (), 'horizontal': ('H',)}.items()):
    col = cond_cols[cond]
    ax.hist(np.rad2deg(df.loc[df['condition'] == cond, 'angle']),
            bins=np.linspace(-180, 180, 37), density=True, color=col, alpha=0.25, label='Waggle runs')
    # Rescale densities from per-radian to per-degree for comparison with the histogram
    for (label, post_m, ls) in [('Unimodal (0)', post_uni, '--'), ('Axial mixture (2)', post_vh, '-')]:
        med, (lo, hi) = density_band(*population_params(post_m, effs))
        ax.plot(theta_deg, med * np.pi / 180, color=col, linestyle=ls, lw=2, label=label)
        ax.fill_between(theta_deg, lo * np.pi / 180, hi * np.pi / 180, color=col, alpha=0.15)
    ax.set_title(f"{cond.capitalize()}, low solar elevation", fontweight='bold')
    ax.set_xlabel('Waggle-run angle relative to feeder (°)')
    ax.legend(loc='upper right')
axes[0].set_ylabel('Density')
plt.tight_layout()
plt.show()

# Possible extensions
- **Random slopes**: add `(1 + H |Individual)` to $\mu$, $\kappa$ and $w$ to follow `HS_circGLMM_fit_4cond.ipynb`, if the 12 individuals in both conditions allow it.
- **Uniform component**: a third, uniform component would represent disoriented waggle runs (as with `M2B`/`M2C` in `circMLE`, which are excluded in `HS_ExploratoryPlots.Rmd`).
- **Free mode separation**: estimate the offset between modes (prior centred on $\pi$) instead of fixing it. This adds a further labelling ambiguity.
- **Separate concentrations** for the two modes.
- **Other conditions**: at low elevations, zenith has 4 individuals (3 also in vertical/horizontal) and 5 dances, so a zenith effect on $w$ would be weakly informed. Antisolar has only one individual (4 dances), so its effects would be informed almost entirely by the prior. Each additional condition can also switch labels independently. (Tilted dances are excluded, since their directions are not meaningfully related to the feeder.)
- **Other elevations**: compare bimodality at low and medium elevations, using the successive-difference coding from `HS_circGLMM_fit_4cond.ipynb`.
- **K-fold cross-validation** leaving out whole dances, if PSIS-LOO diagnostics are poor.